# DES Research Chatbot

By: Maya Fetzer
03/2026

This Google Colab notebook implements a local Retrieval-Augmented Generation (RAG) chatbot designed to extract, structure, and interact with knowledge from research papers for work on Deep Eutectic Solvents (DES).

The system allows users to:

*   Upload one or more research papers (PDF format)
*   Automatically extract and chunk text
*   Generate semantic embeddings
*   Store them in a FAISS vector database
*   Retrieve relevant sections based on user queries
*   Generate context-grounded answers using a local open-source language model

This notebook runs entirely with open-source models and does not require any API key.

https://www.sciencedirect.com/science/article/pii/S2666952825000433#sec7

In [ ]:
# ================================
# INSTALL DEPENDENCIES
# ================================
!pip -q install transformers sentence-transformers faiss-cpu pypdf accelerate

# ================================
# IMPORT LIBRARIES
# ================================
import torch
import numpy as np
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from google.colab import files

# ================================
# DEVICE SETUP
# ================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

# ================================
# STEP 1 — UPLOAD PDF PAPERS
# ================================
print("Upload your research papers (PDF)...")
uploaded = files.upload()

pdf_text = ""

for file_name in uploaded.keys():
    reader = PdfReader(file_name)
    for page in reader.pages:
        if page.extract_text():
            pdf_text += page.extract_text()

print("Total text length:", len(pdf_text))


# ================================
# STEP 2 — CHUNK TEXT
# ================================
chunk_size = 500
overlap = 100

chunks = []
start = 0

while start < len(pdf_text):
    end = start + chunk_size
    chunk = pdf_text[start:end]
    chunks.append(chunk)
    start += chunk_size - overlap

print("Total chunks created:", len(chunks))


# ================================
# STEP 3 — LOAD EMBEDDING MODEL
# ================================
print("Loading embedding model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks)

print("Embeddings created:", embeddings.shape)


# ================================
# STEP 4 — BUILD FAISS VECTOR INDEX
# ================================
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index size:", index.ntotal)


# ================================
# STEP 5 — RETRIEVAL FUNCTION
# ================================
def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(np.array(query_embedding), top_k)

    results = [chunks[i] for i in indices[0]]

    return results


# ================================
# STEP 6 — LOAD LOCAL LLM
# ================================
print("Loading language model...")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

model.to(device)

print("Model loaded successfully")


# ================================
# STEP 7 — CHAT FUNCTION
# ================================
def chat(query):

    context_chunks = retrieve(query)

    context = "\n\n".join(context_chunks)

    prompt = f"""
You are a scientific research assistant.

Use ONLY the provided context from the paper to answer.

If the answer is not present, say:
"The paper does not provide this information."

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response[len(prompt):].strip()

    if answer == "":
        answer = response.strip()

    return answer


# ================================
# STEP 8 — INTERACTIVE CHAT
# ================================
print("\nChatbot ready!")
print("Ask questions about your uploaded papers.\n")

while True:

    query = input("Ask a question (or type 'exit'): ")

    if query.lower() == "exit":
        print("Chat ended.")
        break

    answer = chat(query)

    print("\nAnswer:\n")
    print(answer)
    print("\n" + "-"*60 + "\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 2.5 MB/s eta 0:00:00
Running on: cpu
Upload your research papers (PDF)...


Saving 1-s2.0-S2666952825000433-main.pdf to 1-s2.0-S2666952825000433-main.pdf
Saving deep-eutectic-solvents-(dess)-and-their-applications.pdf to deep-eutectic-solvents-(dess)-and-their-applications.pdf
Saving main.pdf to main.pdf
Total text length: 318184
Total chunks created: 796
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully

Chatbot ready!
Ask questions about your uploaded papers.


Answer:

a deep eutectic solvent (DES) is a type of environmentally benign 
alternative for synthesis that involves the formation of a complex between a 
quaternary ammonium salt and a metal salt. DESs can be prepared by complexing a 
quaternary ammonium salt with a metal salt or hydrogen bond donor. DESs 
are typically obtained by the complexation of a quaternary ammonium salt with a 
metal salt or hydrogen bond donor (HBD).

------------------------------------------------------------

